# Home Credit — Default Risk Prediction (GCI competition)

Reproducible submission notebook: raw CSV files → feature engineering → LightGBM + CatBoost (5-fold stratified cross-validation) → logistic-regression stacking → `output/submission.csv`.

**How to run on Google Colab**
1. Upload the `competition` folder (`train_part1.csv`, `train_part2.csv`, `test.csv`, `sample_submission.csv`) to Google Drive.
2. Edit the `PATH` variable in the next cell so that it points to that folder.
3. Choose *Runtime → Run all*. Expected run time: about 30–60 minutes on a Colab CPU runtime.

**Main feature groups** (none of them uses the `TARGET` label, except the target encoding, which is computed inside each fold only):
- Basic features: credit/income/annuity ratios, `EXT_SOURCE` combinations, deviations from occupation/organization group means.
- `IMP`: model-based imputation of `EXT_SOURCE_1/2/3` with LightGBM regressors.
- `RATE`, `TERM`: implied interest rate solved from the annuity equation, and the reconstructed loan term.
- `PROF`, `PQ`: "loan-product population profile" — average characteristics of all applicants holding the same loan product (term × rate, and × credit-amount quintile).
- `TE_PRODUCT`: nested in-fold target encoding of the loan product (LightGBM); `PRODUCT_CAT`: the loan product as a categorical feature (CatBoost).

**Cross-validated results (OOF ROC-AUC, 4-CPU test run):** LightGBM 0.77051 · CatBoost 0.76870 · Stacking **0.77095**.

In [ ]:
# ===== 1. Data path =====
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PATH = '/content/drive/MyDrive/GCI/competition/'   # <-- change to your own folder
except ImportError:
    PATH = os.environ.get('HC_PATH', './')               # running outside Colab
OUT = os.path.join(PATH, 'output'); os.makedirs(OUT, exist_ok=True)
print('PATH =', PATH)

In [ ]:
# ===== 2. Libraries =====
import subprocess, sys
try:
    import catboost
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'catboost'], check=True)
import numpy as np, pandas as pd, lightgbm as lgb, time, warnings
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')
NJ = os.cpu_count()
SEEDS_LGB = [0, 1]          # LightGBM seeds
SEEDS_CAT = [0]             # CatBoost seeds
print('CPU:', NJ, '| lightgbm', lgb.__version__, '| catboost', catboost.__version__)

In [ ]:
# ===== 3. Load data + basic features =====
rd = lambda f: pd.read_csv(os.path.join(PATH, f), encoding='utf-8-sig')
tr = pd.concat([rd('train_part1.csv'), rd('train_part2.csv')], ignore_index=True)
te = rd('test.csv')
for d in (tr, te): d.columns = [c.replace('﻿', '') for c in d.columns]
y = tr.pop('TARGET').values; n = len(tr)
df = pd.concat([tr, te], ignore_index=True)
print('train', tr.shape, 'test', te.shape, 'default rate', round(y.mean(), 4))

df['DAYS_EMPLOYED_ANOM'] = (df.DAYS_EMPLOYED == 365243).astype(int); df.loc[df.DAYS_EMPLOYED == 365243, 'DAYS_EMPLOYED'] = np.nan
E = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']; e = df[E]
df['EXT_MEAN'] = e.mean(1); df['EXT_STD'] = e.std(1); df['EXT_PROD'] = e.prod(1, min_count=3); df['EXT_MIN'] = e.min(1); df['EXT_MAX'] = e.max(1)
df['EXT_W'] = df.EXT_SOURCE_1.fillna(df.EXT_MEAN)*2 + df.EXT_SOURCE_2.fillna(df.EXT_MEAN) + df.EXT_SOURCE_3.fillna(df.EXT_MEAN)*3
df['EXT_NA'] = e.isna().sum(1); df['EXT_HMEAN'] = 3/(1/e).sum(1, min_count=1)
for a in E: df[a+'_x_AGE'] = df[a]*df.DAYS_BIRTH
df['E1E2'] = df.EXT_SOURCE_1*df.EXT_SOURCE_2; df['E2E3'] = df.EXT_SOURCE_2*df.EXT_SOURCE_3; df['E1E3'] = df.EXT_SOURCE_1*df.EXT_SOURCE_3
df['CREDIT_ANNUITY'] = df.AMT_CREDIT/df.AMT_ANNUITY; df['CREDIT_GOODS'] = df.AMT_CREDIT/df.AMT_GOODS_PRICE
df['GOODS_MINUS'] = df.AMT_GOODS_PRICE - df.AMT_CREDIT
df['ANNUITY_INCOME'] = df.AMT_ANNUITY/df.AMT_INCOME_TOTAL; df['CREDIT_INCOME'] = df.AMT_CREDIT/df.AMT_INCOME_TOTAL
df['INC_PER_PERSON'] = df.AMT_INCOME_TOTAL/df.CNT_FAM_MEMBERS; df['INC_PER_CHILD'] = df.AMT_INCOME_TOTAL/(1+df.CNT_CHILDREN)
for c in ['DAYS_EMPLOYED', 'OWN_CAR_AGE', 'DAYS_LAST_PHONE_CHANGE', 'DAYS_ID_PUBLISH', 'DAYS_REGISTRATION']: df[c+'_R_BIRTH'] = df[c]/df.DAYS_BIRTH
df['EXT_x_CA'] = df.EXT_MEAN*df.CREDIT_ANNUITY; df['EXT_d_CA'] = df.EXT_MEAN/df.CREDIT_ANNUITY
df['ANN_LEFT'] = df.AMT_INCOME_TOTAL - df.AMT_ANNUITY
df['REQ_SUM'] = df[[c for c in df.columns if c.startswith('AMT_REQ')]].sum(1)
df['TERM_ROUND'] = np.round(df.CREDIT_ANNUITY)
cats = df.select_dtypes(exclude='number').columns.tolist()
for c in cats:
    df[c] = df[c].fillna('NA').astype(str); df[c+'_FREQ'] = df[c].map(df[c].value_counts(normalize=True))
for c in ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE']:
    g = df.groupby(c)
    for v in ['EXT_MEAN', 'AMT_INCOME_TOTAL', 'CREDIT_ANNUITY', 'DAYS_BIRTH', 'ANNUITY_INCOME']:
        df[f'{c}_{v}_DIFF'] = df[v] - g[v].transform('mean')
df['INC_ORG_OCC_REL'] = df.AMT_INCOME_TOTAL/df.groupby(['ORGANIZATION_TYPE', 'OCCUPATION_TYPE']).AMT_INCOME_TOTAL.transform('median')
df = df.replace([np.inf, -np.inf], np.nan)
ids = df.SK_ID_CURR.values; df = df.drop(columns='SK_ID_CURR')
print('after basic features:', df.shape, '| categorical columns:', len(cats))

In [ ]:
# ===== 4. IMP: model-based imputation of EXT_SOURCE (no TARGET used) =====
t0 = time.time()
d = df.copy()
for c in cats: d[c] = d[c].astype('category')
EXTD = [c for c in d.columns if 'EXT' in c or c.startswith('E1') or c.startswith('E2')]
base = [c for c in d.columns if c not in EXTD]
Pr = dict(objective='regression', learning_rate=0.05, num_leaves=31, feature_fraction=0.5, min_child_samples=50, verbose=-1, n_jobs=NJ)
new = {}
for tgt in E:
    cols = base + [x for x in E if x != tgt]; have = d[tgt].notna().values; pred = np.zeros(len(d))
    hv = np.where(have)[0]; nh = np.where(~have)[0]
    for a, b in KFold(5, shuffle=True, random_state=0).split(hv):
        m = lgb.train(Pr, lgb.Dataset(d.iloc[hv[a]][cols], d[tgt].values[hv[a]]), 800)
        pred[hv[b]] = m.predict(d.iloc[hv[b]][cols]); pred[nh] += m.predict(d.iloc[nh][cols])/5
    print(tgt, 'prediction-vs-actual correlation', round(np.corrcoef(pred[hv], d[tgt].values[hv])[0, 1], 3), f'{time.time()-t0:.0f}s')
    new[tgt+'_PRED'] = pred; new[tgt+'_FILL'] = df[tgt].fillna(pd.Series(pred)).values; new[tgt+'_RES'] = (df[tgt] - pred).values
del d
F = pd.DataFrame({x: new[x+'_FILL'] for x in E})
new['EXTF_MEAN'] = F.mean(1).values; new['EXTF_PROD'] = F.prod(1).values; new['EXTF_MIN'] = F.min(1).values; new['EXTF_MAX'] = F.max(1).values
new['EXTF_STD'] = F.std(1).values; new['EXTF_W'] = (F.EXT_SOURCE_1*2 + F.EXT_SOURCE_2 + F.EXT_SOURCE_3*3).values
G = {'IMP': list(new)}

In [ ]:
# ===== 5. RATE + TERM: implied interest rate and reconstructed loan term =====
def rate(C, A, nn):
    """Monthly rate r solving A = C*r/(1-(1+r)^-n) by bisection; NaN if A*n <= C."""
    lo = np.full_like(C, 1e-7); hi = np.full_like(C, 0.25)
    for _ in range(60):
        mid = (lo+hi)/2; pv = A*(1-(1+mid)**-nn)/mid
        lo = np.where(pv > C, mid, lo); hi = np.where(pv > C, hi, mid)
    r = (lo+hi)/2; r[~(A*nn > C)] = np.nan; return r

C = df.AMT_CREDIT.values.astype(float); A = df.AMT_ANNUITY.values.astype(float)
rev = (df.NAME_CONTRACT_TYPE != 'Cash loans').values
TL = [6, 10, 12, 18, 24, 30, 36, 42, 48, 54, 60]; rates = {}
for nn in TL:
    r = rate(C, A, nn)*12; r[rev] = np.nan; rates[nn] = r
G['RATE'] = []
for nn in [12, 24, 36, 48, 60]: new[f'RATE_{nn}'] = rates[nn]; G['RATE'].append(f'RATE_{nn}')
# term = the candidate term whose implied rate is most frequent (densest) across train+test
dens = []
for nn in TL:
    key = pd.Series(np.round(rates[nn]*400)); cnt = key.map(key.value_counts()).fillna(0).values.astype(float)
    cnt[np.isnan(rates[nn])] = 0; cnt[rates[nn] > 0.6] = 0; dens.append(cnt)
D = np.vstack(dens).T; j = D.argmax(1); srt = np.sort(D, 1)
th = np.array(TL, float)[j]; rh = np.vstack([rates[t] for t in TL]).T[np.arange(len(j)), j]
th[rev | (D.max(1) == 0)] = np.nan; rh[np.isnan(th)] = np.nan
new['TERM_HAT'] = th; new['RATE_HAT'] = rh; new['TERM_DENS'] = np.where(np.isnan(th), np.nan, srt[:, -1])
new['TERM_DENS_RATIO'] = np.where(np.isnan(th), np.nan, srt[:, -1]/(srt[:, -2]+1))
new['OVERPAY_REL'] = A*th/C - 1; new['AGE_END_LOAN'] = -df.DAYS_BIRTH.values/365.25 + th/12
prod = pd.Series(np.round(C/A, 2)); new['PRODUCT_FREQ'] = prod.map(prod.value_counts()).values.astype(float)
pg = pd.Series(np.round(C/df.AMT_GOODS_PRICE.values, 3)); new['CG_FREQ'] = pg.map(pg.value_counts()).values.astype(float)
G['TERM'] = ['TERM_HAT', 'RATE_HAT', 'TERM_DENS', 'TERM_DENS_RATIO', 'OVERPAY_REL', 'AGE_END_LOAN', 'PRODUCT_FREQ', 'CG_FREQ']
print('share of loans with a reconstructed term:', round(np.mean(~np.isnan(th)), 3))

In [ ]:
# ===== 6. PROF + PQ: loan-product population profile (no TARGET used) =====
def profile(key, cols, pre, minc=30, std=True):
    p = pd.Series(key).fillna(-9).values; d = cols.copy(); d['p'] = p; s = pd.Series(p); cnt = s.map(s.value_counts()).values
    g = d.groupby('p'); o = pd.DataFrame({f'{pre}_{c}': g[c].transform('mean').values for c in cols.columns})
    if std: o[pre+'_estd'] = g['e'].transform('std').values
    o[cnt < minc] = np.nan; return o

cash = ~rev
core = pd.DataFrame({'e': new['EXTF_MEAN'], 'e2': df.EXT_SOURCE_2.values, 'e3': df.EXT_SOURCE_3.values, 'e1p': new['EXT_SOURCE_1_PRED'],
                     'age': df.DAYS_BIRTH.values, 'emp': df.DAYS_EMPLOYED.values, 'male': (df.CODE_GENDER == 'M').values.astype(float)})
kTR = pd.Series(np.where(cash, th*100000 + np.round(rh*200), -5)).fillna(-9).values
p = profile(kTR, core, 'PT'); G['PROF'] = list(p.columns)
for c in p.columns: new[c] = p[c].values
kPQ = kTR*10 + pd.qcut(df.AMT_CREDIT, 5, labels=False).values
p = profile(kPQ, core[['e', 'e3', 'age', 'emp']], 'PQ', std=False); G['PQ'] = list(p.columns)
for c in p.columns: new[c] = p[c].values
df = pd.concat([df, pd.DataFrame(new)], axis=1).replace([np.inf, -np.inf], np.nan)
PROD = np.round(df.AMT_CREDIT/df.AMT_ANNUITY, 2).astype(str).values      # loan-product key
print('total columns:', df.shape[1], {k: len(v) for k, v in G.items()})

In [ ]:
# ===== 7. Nested in-fold target encoding of the loan product (no label leakage) =====
def te_map(keys, yy, idx_fit, idx_apply, alpha=10):
    pr = yy[idx_fit].mean(); d = pd.DataFrame({'k': keys[idx_fit], 'y': yy[idx_fit]}).groupby('k').y.agg(['sum', 'count'])
    v = (d['sum'] + alpha*pr)/(d['count'] + alpha); return pd.Series(keys[idx_apply]).map(v).fillna(pr).values

def nested_te(keys, yy, a, apply_list, alpha=10, seed=0):
    """Training part a: encoded with 5 inner folds within a. Valid/test: fitted on all of a. Labels outside a are never used."""
    ta = np.zeros(len(a))
    for ia, ib in StratifiedKFold(5, shuffle=True, random_state=seed).split(a, yy[a]): ta[ib] = te_map(keys, yy, a[ia], a[ib], alpha)
    return ta, [te_map(keys, yy, a, idx, alpha) for idx in apply_list]

newc = set(sum(G.values(), [])); orig = [c for c in df.columns if c not in newc]
folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(n), y))

In [ ]:
# ===== 8. LightGBM (features + PQ + TE_PRODUCT) =====
cols_l = orig + G['IMP'] + G['RATE'] + G['TERM'] + G['PROF'] + G['PQ']
X = df.iloc[:n][cols_l].reset_index(drop=True); T = df.iloc[n:][cols_l].reset_index(drop=True)
for c in cats: X[c] = X[c].astype('category'); T[c] = pd.Categorical(T[c], categories=X[c].cat.categories)
P = dict(objective='binary', learning_rate=0.01, num_leaves=16, max_depth=6, min_child_samples=100, feature_fraction=0.3, bagging_fraction=0.8,
         bagging_freq=1, lambda_l1=2, lambda_l2=10, min_gain_to_split=0.01, cat_smooth=50, cat_l2=20, max_cat_to_onehot=4, verbose=-1, n_jobs=NJ)
oof_lgb = np.zeros(n); pt_lgb = np.zeros(len(T)); t0 = time.time()
for f, (a, b) in enumerate(folds):
    Xa, Xb, Tt = X.iloc[a].reset_index(drop=True), X.iloc[b].reset_index(drop=True), T.copy()
    ta, (tb, tt) = nested_te(PROD, y, a, [b, np.arange(n, len(df))]); Xa['TE_PRODUCT'] = ta; Xb['TE_PRODUCT'] = tb; Tt['TE_PRODUCT'] = tt
    for s in SEEDS_LGB:
        m = lgb.train({**P, 'seed': s}, lgb.Dataset(Xa, y[a]), 10000, valid_sets=[lgb.Dataset(Xb, y[b])], callbacks=[lgb.early_stopping(300, verbose=False)])
        oof_lgb[b] += m.predict(Xb, num_iteration=m.best_iteration)/len(SEEDS_LGB)
        pt_lgb += m.predict(Tt, num_iteration=m.best_iteration)/len(SEEDS_LGB)/5
    print(f'LGB fold {f}: AUC {roc_auc_score(y[b], oof_lgb[b]):.5f}  ({time.time()-t0:.0f}s)')
print('LightGBM OOF AUC:', round(roc_auc_score(y, oof_lgb), 5))

In [ ]:
# ===== 9. CatBoost (features + PRODUCT_CAT) =====
cols_c = orig + G['IMP'] + G['RATE'] + G['TERM'] + G['PROF']
vc = pd.Series(PROD).value_counts(); df['PRODUCT_CAT'] = np.where(pd.Series(PROD).map(vc).values < 30, 'rare', PROD)
cols_c = cols_c + ['PRODUCT_CAT']; cats_c = cats + ['PRODUCT_CAT']
X = df.iloc[:n][cols_c].reset_index(drop=True); T = df.iloc[n:][cols_c].reset_index(drop=True)
oof_cat = np.zeros(n); pt_cat = np.zeros(len(T)); t0 = time.time()
for f, (a, b) in enumerate(folds):
    for s in SEEDS_CAT:
        m = CatBoostClassifier(iterations=5000, learning_rate=0.05, depth=6, l2_leaf_reg=5, eval_metric='AUC', od_type='Iter', od_wait=200,
                               thread_count=NJ, verbose=0, random_seed=s)
        m.fit(X.iloc[a], y[a], cat_features=cats_c, eval_set=(X.iloc[b], y[b]), use_best_model=True)
        oof_cat[b] += m.predict_proba(X.iloc[b])[:, 1]/len(SEEDS_CAT)
        pt_cat += m.predict_proba(T)[:, 1]/len(SEEDS_CAT)/5
    print(f'Cat fold {f}: AUC {roc_auc_score(y[b], oof_cat[b]):.5f}  ({time.time()-t0:.0f}s)')
print('CatBoost OOF AUC:', round(roc_auc_score(y, oof_cat), 5))

In [ ]:
# ===== 10. Logistic-regression stacking on prediction logits =====
lg = lambda p: np.log(np.clip(p, 1e-6, 1-1e-6)/(1-np.clip(p, 1e-6, 1-1e-6)))
M = np.c_[lg(oof_lgb), lg(oof_cat)]; Mt = np.c_[lg(pt_lgb), lg(pt_cat)]
oof_st = np.zeros(n)
for a, b in folds:                                   # nested evaluation: meta-model fitted on 4 folds, scored on the held-out fold
    oof_st[b] = LogisticRegression(C=1.0).fit(M[a], y[a]).decision_function(M[b])
meta = LogisticRegression(C=1.0).fit(M, y); pt_st = meta.decision_function(Mt)
print('stacking coefficients (LGB, Cat):', np.round(meta.coef_[0], 3))
print('OOF AUC — LightGBM %.5f | CatBoost %.5f | Stacking %.5f' % (roc_auc_score(y, oof_lgb), roc_auc_score(y, oof_cat), roc_auc_score(y, oof_st)))
print('per-fold AUC (stacking):', [round(roc_auc_score(y[b], oof_st[b]), 5) for a, b in folds])

In [ ]:
# ===== 11. Write the submission file =====
from scipy.stats import rankdata
ss = rd('sample_submission.csv'); ss.columns = [c.replace('﻿', '') for c in ss.columns]
assert (ss.SK_ID_CURR.values == ids[n:]).all()
sub = pd.DataFrame({'SK_ID_CURR': ss.SK_ID_CURR.values, 'TARGET': rankdata(pt_st)/len(pt_st)})
assert len(sub) == len(ss) and sub.TARGET.notna().all()
sub.to_csv(os.path.join(OUT, 'submission.csv'), index=False)
print('Saved', os.path.join(OUT, 'submission.csv'), sub.shape); sub.head()

In [ ]:
# ===== 12. Package the code for submission (Competition .zip) =====
# On Colab: save the notebook (Ctrl+S) before running this cell. It zips the notebooks in PATH + output/submission.csv into code.zip.
import zipfile, glob
zp = os.path.join(PATH, 'code_v2.zip')
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(os.path.join(PATH, '*.ipynb')): z.write(f, os.path.basename(f))
    z.write(os.path.join(OUT, 'submission.csv'), 'output/submission.csv')
print('Created', zp, '- download it and submit it under "Competition (.zip)" on Omnicampus.')